In [ ]:
try:
    from pydub import AudioSegment
except ImportError:
    %pip install pydub
    %pip install pydub[extras]
    from pydub import AudioSegment
    from pydub.playback import play


In [ ]:
from IPython.display import Audio
audio_path = '../datasets/audio/Prime-minister.m4a'
audio = AudioSegment.from_file(audio_path, format="m4a")
audio

In [ ]:
try:    
    import whisper
except ImportError:
    %pip install openai-whisper
    import whisper

In [ ]:
whisper_model = whisper.load_model("base.en")


In [ ]:
transcription = whisper_model.transcribe(audio_path, fp16=True, verbose=False)

In [ ]:
transcription

In [ ]:
from IPython.display import Audio

In [ ]:
try:
    from gtts import gTTS
except ImportError:
    %pip install gtts
    from gtts import gTTS

In [ ]:
def speak(text, file):
    tts = gTTS(text, lang='en')
    with open(file, 'wb') as f:
        tts.write_to_fp(f)
    return Audio(file)

In [ ]:
speak(transcription['text'], '../datasets/audio/pm-2.mp3')

In [ ]:
from langchain.llms import Ollama
from langchain.callbacks.manager import CallbackManager
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler 

In [ ]:
llm = Ollama(model="llama2", 
             callback_manager = CallbackManager([StreamingStdOutCallbackHandler()]))

In [ ]:
def answers(llm, prompt_qs, prompts, text):
    outputs = []
    for prompt, prompt_qs in zip(prompts, prompt_qs):
        print(prompt_qs, end="\n")
        output = llm(prompt, temperature=0.5)
        #print(output, end="\n\n")
        print("\n" + "=="*50, end="\n\n")
    outputs.append(output) 
    return outputs

In [ ]:
prompt_qs = ["Please be concise."] 
prompts = [q + ":"+ transcription["text"] for q in prompt_qs]

outputs = answers(llm, prompt_qs, prompts, transcription["text"])

In [ ]:
speak(outputs[0].replace("\n", ""), '../datasets/audio/pm-answer.mp3')

References

1. [Whisper](https://blog.devgenius.io/transcribing-youtube-videos-using-openais-whisper-%EF%B8%8F-%EF%B8%8F-a29d264d6fb1)
2. [Langchain and LLama](https://www.youtube.com/watch?v=k_1pOF1mj8k)
3. [English to Hindi using Transformers](https://prateekjoshi.medium.com/english-to-hindi-translation-made-simple-with-transformers-library-33f64f745552)

In [ ]:
from IPython.display import YouTubeVideo
YouTubeVideo('https://www.youtube.com/watch?v=CuBzyh4Xmvk', width=500, height=300)

In [ ]:
try:
    import yt_dlp
except ImportError:
    %pip install yt_dlp
    import yt_dlp

In [ ]:
def download(video_id: str, save_path: str) -> str:
    video_url = f'https://www.youtube.com/watch?v={video_id}'
    ydl_opts = {
        'format': 'm4a/bestaudio/best',
        'paths': {'home': save_path},
        'outtmpl': {'default': "lecture.m4a"},
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'm4a',
        }]
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        error_code = ydl.download([video_url])
        if error_code != 0:
            raise Exception('Failed to download video')

    return save_path

In [ ]:
download('CuBzyh4Xmvk', '../datasets/audio/')

In [ ]:
audio_path = '../datasets/audio/lecture.m4a'
audio = AudioSegment.from_file(audio_path, format="m4a")

In [ ]:
audio[:13000]

In [ ]:
transcription = whisper_model.transcribe("../datasets/audio/lecture.m4a", fp16=True, verbose=False)

In [ ]:
print(transcription["text"][:500].replace(". ", "\n"))

In [ ]:
transcription.keys()

In [ ]:
def create_srt_from_transcription(transcription_objects, srt_file_path):
    with open(srt_file_path, 'w') as srt_file:
        index = 1  # SRT format starts with index 1

        for entry in transcription_objects['segments']:
            start_time = entry['start']
            end_time = entry['end']
            text = entry['text']

            # Convert time to SRT format
            start_time_str = format_time(start_time)
            end_time_str = format_time(end_time)

            # Write entry to SRT file
            srt_file.write(f"{index}\n")
            srt_file.write(f"{start_time_str} --> {end_time_str}\n")
            srt_file.write(f"{text}\n\n")

            index += 1

def format_time(time_seconds):
    minutes, seconds = divmod(time_seconds, 60)
    hours, minutes = divmod(minutes, 60)
    return f"{int(hours):02d}:{int(minutes):02d}:{int(seconds):02d},000"


In [ ]:
create_srt_from_transcription(transcription, "../datasets/audio/lecture.srt")

In [ ]:
!head ../datasets/audio/lecture.srt

In [ ]:
speak(transcription['text'][:1300], '../datasets/audio/hello.mp3')

In [ ]:
try:
    from transformers import MBartForConditionalGeneration, MBart50TokenizerFast
except:
    %pip install transformers -U -q
    %pip install sentencepiece
    from transformers import MBartForConditionalGeneration, MBart50TokenizerFast
    

In [ ]:

# download and save model
model = MBartForConditionalGeneration.from_pretrained("facebook/mbart-large-50-one-to-many-mmt")

# import tokenizer
tokenizer = MBart50TokenizerFast.from_pretrained("facebook/mbart-large-50-one-to-many-mmt", src_lang="en_XX")


In [ ]:
text_to_translate = transcription["text"][:500].split(". ")
text_to_translate

In [ ]:
model_inputs = tokenizer(text_to_translate, return_tensors="pt", padding=True, truncation=True)

In [ ]:
generated_tokens = model.generate(
    **model_inputs,
    forced_bos_token_id=tokenizer.lang_code_to_id["hi_IN"]
)

translation = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)



In [ ]:
translation

In [ ]:
llm = Ollama(model="mistral", 
             callback_manager = CallbackManager([StreamingStdOutCallbackHandler()]))
prompt_qs = ["Please provide a bullet-point summary for the given text:",
             "Highlight the important topics and subtopics in the given lecture:",
             "Give us some question for a quiz based on the following text:",
             "Summarize the following text in Hindi in 10 lines or less:",
            ]

prompts = [q + "\n\n" + transcription["text"] for q in prompt_qs]

outputs = answers(llm, prompt_qs, prompts, transcription["text"])